In [59]:
import numpy as np
from numpy.fft import fftfreq, fft, ifft
import matplotlib.pyplot as plt

def plot_plot(*data_pairs, xlabel='x', ylabel='y', scale='linear', title='', xlim=None, ylim=None):
    fig = plt.figure()
    ax = fig.add_subplot(1, 1, 1)
    for pair in data_pairs:
        if len(pair) == 1:
            y, label = pair[0], ''
            x = np.arange(len(y))
        elif len(pair) == 2:
            y, label = pair[0], pair[1]
            x = np.arange(len(y))
        else:
            x, y, label = pair[0], pair[1], pair[2]
        yerr = pair[3] if len(pair) > 3 else None
        fmt = pair[4] if len(pair) > 4 else '-'
        ax.errorbar(x, y, yerr=yerr, fmt=fmt, capsize=5, label=label)
    
    if scale == 'logx':
        ax.set_xscale('log')
    elif scale == 'logy':
        ax.set_yscale('log')
    elif scale == 'loglog':
        ax.set_xscale('log')
        ax.set_yscale('log')
    
    if xlim: ax.set_xlim(xlim)
    if ylim: ax.set_ylim(ylim)

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()

def compute_psd(signal, t, return_spectrum=False):
    dt = t[1] - t[0]
    N = len(t)
    frequency = fftfreq(N, d=dt)
    spectrum = fft(signal)
    psd = dt / N * np.abs(spectrum) ** 2
    mask = frequency >= 0
    if return_spectrum:
        return frequency[mask], psd[mask], spectrum
    return frequency[mask], psd[mask]
    

In [60]:
data = np.loadtxt('luftsensor_studie8.txt', skiprows=1, delimiter=',')
duration = data[:,0]
sensor = data[:,1]
referenz = data[:,2]

unique_durations = np.unique(duration)
y_vals = []

for d in unique_durations:
    mask = (duration == d)
    total = np.sum(mask)
    correct = np.sum(sensor[mask] == referenz[mask])
    y_vals.append(correct / total)

b_fit, ln_a = np.polyfit(unique_durations, np.log(1 - np.array(y_vals)), 1)
a_fit = np.exp(ln_a)

print(f"a = {a_fit:.5f}")
print(f"b = {b_fit:.5f} 1/min")

a = 0.58379
b = -0.44769 1/min


In [61]:
def accuracy_func(t):
    return 1 - a_fit * np.exp(b_fit * t)

print(f"Accuracy at 3 min = {accuracy_func(3) * 100:.5f}%")

def get_time_for_acc(acc):
    return (np.log((1 - acc) / a_fit)) / b_fit

print(f"Time needed for 85% acc = {get_time_for_acc(0.85):.5f} min")

Accuracy at 3 min = 84.76038%
Time needed for 85% acc = 3.03540 min


In [62]:
t = 3.0
p = accuracy_func(t)

N = 2000

sigma_y = np.sqrt(p * (1 - p) / N)
print(f"sigma = {sigma_y:.5f}")

sigma = 0.00804


In [63]:
untere_prozent = (p - 2*sigma_y) * 100
obere_prozent = (p + 2*sigma_y) * 100
print(f"Untere Grenze = {untere_prozent:.5f}%")
print(f"Obere Grenze = {obere_prozent:.5f}%")

Untere Grenze = 83.15307%
Obere Grenze = 86.36769%


In [64]:
x_val = 3.0
sigma_x = 0.2

dy_dx = -a_fit * b_fit * np.exp(b_fit * x_val)

print(f"{dy_dx * sigma_x * 100:.5f}%")


1.36451%


In [65]:
p_K = 0.25
t = 3.0
p_M = accuracy_func(t)

p_K_M = (p_M * p_K) / ((p_M * p_K) + ((1 - p_M) * (1 - p_K)))

print(f"Warscheinlichkeit einer Kritischne Belastung: {p_K_M * 100:.5f}%")

Warscheinlichkeit einer Kritischne Belastung: 64.96082%


In [66]:
p_K = p_K_M
t = 3.0
p_M = accuracy_func(t)

p_K_M_2 = (p_M * p_K) / ((p_M * p_K) + ((1 - p_M) * (1 - p_K)))

print(f"Warscheinlichkeit einer Kritischne Belastung: {p_K_M_2 * 100:.5f}%")

Warscheinlichkeit einer Kritischne Belastung: 91.15934%
